In [91]:
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PowerTransformer
from xgboost import XGBClassifier
import sqlalchemy as sql
import pandas as pd
import numpy as np
import urllib
import re

#### Data Loading

In [121]:
products_df     = pd.read_csv('Data/products.csv')
customers_df    = pd.read_csv('Data/customers.csv')
transactions_df = pd.read_csv('Data/transactions.csv')
sessions_df     = pd.read_csv('Data/sessions.csv')
labels_df       = pd.read_csv('Data/labels.csv')

In [122]:
products_df.head(1)

,product_id,product_name,category,price,brand,rating
0,P0001,Product_1,Sports,446,BrandC,3.8


In [123]:
customers_df.head(1)

,customer_id,age,gender,education,job,annual_income,city,marital_status
0,C0002,46,Female,High School,Sales,9970,Bandung,Single


In [124]:
transactions_df.head(1)

,transaction_id,customer_id,product_id,transaction_date,quantity,total_amount,payment_method
0,T00001,C0108,P0044,2025-01-01 00:00:00,1,607,E-Wallet


In [125]:
sessions_df.head(1)

,session_id,customer_id,session_date,device,traffic_source,pages_viewed,time_spent_sec,product_id
0,S00632,C0049,2025-01-27 07:00:00,Desktop,Social,1,200,P0010


In [126]:
labels_df.head(1)

,session_id,purchase
0,S00632,0


#### Feature Engineering

**Menambahkan history jumlah transaksi di setiap kategori produk sebelum transaksi**

In [93]:
# Tabel transaksi join dengan tabel produk
df1 = transactions_df.merge(
                            products_df[['product_id','category']], 
                            on = "product_id", 
                            how = "left"
                        )

# History Transaksi tiap Tipe Produk
for i in range(df1.shape[0]) :
    
    # Mengambil data histori dari customer
    history_df = df1[(
                        df1['customer_id'] == df1.loc[i, 'customer_id']
                    ) & (df1['transaction_date'] < df1.loc[i, 'transaction_date'])]
    
    # Menjumlahkan total transaksi sebelum suatu transaksi dilakukan
    df1.loc[i, f'num_prev_transaction'] = np.sum(history_df['total_amount'])
    
    # Mengambil jumlah transaksi yang pernah dilakukan dari setiap kategorinya
    for prod_type in df1['category'].unique() :
        df1.loc[i, f'{prod_type}_transaction_history'] = history_df[history_df['category'] == prod_type].shape[0]

In [94]:
df1.head(2)

,transaction_id,customer_id,product_id,transaction_date,quantity,total_amount,payment_method,category,num_prev_transaction,Fashion_transaction_history,Home_transaction_history,Electronics_transaction_history,Sports_transaction_history
0,T00001,C0108,P0044,2025-01-01 00:00:00,1,607,E-Wallet,Fashion,0.0,0.0,0.0,0.0,0.0
1,T00002,C0074,P0009,2025-01-01 01:00:00,1,657,E-Wallet,Home,0.0,0.0,0.0,0.0,0.0


**Menambahkan history total transaksi sebelum dan transaksi terakhir sebelum dilakukan sesi**

In [95]:
# Tabel Session join dengan tabel label
df2 = sessions_df.merge(labels_df, on = "session_id").copy()

# History transaksi customer ketika sessions dilakukan
for i in range(df2.shape[0]) :
    history_df = df1[(
                        df1['customer_id'] == df2.loc[i, 'customer_id']
                    ) & (df1['transaction_date'] < df2.loc[i, 'session_date'])].copy()

    # Pembuatan kolom yang berisi jumlah transaksi yang pernah dilakukan sebelum sessions
    if not history_df.empty:
        num_prev_tr = history_df.tail(1)[f'num_prev_transaction'].values[0]
    else:
        num_prev_tr = 0
    
    df2.loc[i, f'num_prev_transaction'] = int(num_prev_tr)
    
    # Pembuatan kolom yang berisi histori terakhir customer pada setiap kategori produk
    for prod_type in df1['category'].unique() :
        if not history_df.empty:
            prev_tr_typ = history_df.tail(1)[f'{prod_type}_transaction_history'].values[0]
        else:
            prev_tr_typ = 0

        df2.loc[i, f'{prod_type}_transaction_history'] = int(prev_tr_typ)

In [97]:
df2.head(2)

,session_id,customer_id,session_date,device,traffic_source,pages_viewed,time_spent_sec,product_id,purchase,num_prev_transaction,Fashion_transaction_history,Home_transaction_history,Electronics_transaction_history,Sports_transaction_history
0,S00632,C0049,2025-01-27 07:00:00,Desktop,Social,1,200,P0010,0,1838.0,0.0,1.0,2.0,0.0
1,S01468,C0068,2025-03-03 03:00:00,Desktop,Organic,3,63,P0050,0,4798.0,2.0,0.0,0.0,3.0


In [98]:
# Penggabungan data sessions dengan keterangan yang ada di produk dan customer
df3 = df2.copy()
df3 = df3.merge(products_df[['product_id', 'category']], on = 'product_id')
df3 = df3.merge(customers_df, on = 'customer_id').drop(columns = [
        'session_id', 'customer_id', 'session_date', "product_id"
])

In [99]:
# Pembagian Data berdasarkan tipe data
nominal_cat = df3.select_dtypes("object").drop(columns = ['education'])
ordinal_cat = df3['education'].map({
                            "High School" : 0,
                            "Bachelor"    : 1,
                            "Master"      : 2
})

cat_col = pd.concat([pd.get_dummies(nominal_cat), ordinal_cat], axis = 1)
num_col = df3.select_dtypes("number")

C:\Users\user\AppData\Local\Temp\ipykernel_7068\804140710.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  nominal_cat = df3.select_dtypes("object").drop(columns = ['education'])


In [100]:
# Penggabungan kembali data yang telah dibagi
data = pd.concat([cat_col, num_col], axis = 1)

# Penghapusan kolom yang berisi maksud yang sama
data.drop(columns = ['device_Desktop',
                     'gender_Female',
                     'marital_status_Single'], inplace = True)

In [101]:
data

,device_Mobile,traffic_source_Ads,traffic_source_Organic,traffic_source_Social,category_Electronics,category_Fashion,category_Home,category_Sports,gender_Male,job_Engineer,...,pages_viewed,time_spent_sec,purchase,num_prev_transaction,Fashion_transaction_history,Home_transaction_history,Electronics_transaction_history,Sports_transaction_history,age,annual_income
0,False,False,False,True,True,False,False,False,False,False,...,1,200,0,1838.0,0.0,1.0,2.0,0.0,52,7797
1,False,False,True,False,False,False,True,False,True,False,...,3,63,0,4798.0,2.0,0.0,0.0,3.0,25,10519
2,False,False,False,True,False,True,False,False,False,False,...,1,336,0,2009.0,0.0,0.0,1.0,1.0,33,13122
3,False,False,False,True,False,False,False,True,False,False,...,3,483,0,0.0,0.0,0.0,0.0,0.0,40,17229
4,False,True,False,False,False,False,False,True,True,False,...,5,205,0,0.0,0.0,0.0,0.0,0.0,43,11503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
695,True,True,False,False,False,True,False,False,False,True,...,5,286,1,1108.0,1.0,1.0,0.0,0.0,19,15081
696,True,False,True,False,True,False,False,False,True,False,...,4,164,1,1689.0,2.0,1.0,0.0,0.0,40,8722
697,False,False,True,False,True,False,False,False,False,True,...,2,265,1,0.0,0.0,0.0,0.0,0.0,41,14182
698,False,False,True,False,True,False,False,False,True,False,...,3,367,1,938.0,0.0,0.0,0.0,1.0,45,19939


#### Splitting Data

In [102]:
X = data.drop(columns = ['purchase',])
y = data.purchase

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify = y, random_state = 10, test_size = 0.2)

In [103]:
y_train.value_counts()

purchase
0    280
1    280
Name: count, dtype: int64

In [104]:
y_test.value_counts()

purchase
0    70
1    70
Name: count, dtype: int64

#### Cek Outlier Data

In [105]:
def outlier_check(data) :
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)
    iqr = q3 - q1
    outl = len([x for x in data if (x < q1 - 1.5 * iqr) or (x > q3 + 1.5 * iqr)])
    return outl

In [106]:
outlier_df = pd.DataFrame()
num_col = X_train.select_dtypes("number")

for nc in num_col.columns :
    outlier_df.loc[nc, 'count'] = outlier_check(num_col[nc])

In [107]:
outlier_df

,count
education,0.0
pages_viewed,0.0
time_spent_sec,0.0
num_prev_transaction,26.0
Fashion_transaction_history,103.0
Home_transaction_history,137.0
Electronics_transaction_history,6.0
Sports_transaction_history,109.0
age,0.0
annual_income,0.0


In [108]:
# Mengambil daftar kolom numerik yang mengandung dan tidak mengandung outlier
with_outlier    = outlier_df[outlier_df["count"] > 0].index
without_outlier = outlier_df[~outlier_df.index.isin(with_outlier)].index

scaler = {}
power = {}

# RobustScaler digunakan untuk menscaling data yang mengandung outlier karena berbasis median (tahan outlier)
# MinMaxScaler digunakan untuk menscaling data yang tidak mengandung outlier karena sensitif pada outlier
for nc in num_col.columns :
    
    if np.min(X_train[nc]) <= 0 :
        method = 'yeo-johnson'
    else :
        method = 'box-cox'
    
    power[nc] = PowerTransformer(method = method)
        
    if nc in with_outlier :
        scaler[nc] = RobustScaler()
    else:
        scaler[nc] = MinMaxScaler()
    
    # Mentransformasi data dengan setiap scalernya
    # Train --> Mengambil distribusi dari nilai dan bertransformasi menggunakan scaling yang telah melihat distribusi nilai
    # Test  --> Bertransformasi menggunakan scaler yang telah mempelajari distribusi nilai dari data train
    X_train[nc] = scaler[nc].fit_transform(power[nc].fit_transform(X_train[nc].values.reshape(-1, 1)))
    X_test[nc]  = scaler[nc].transform(power[nc].transform(X_test[nc].values.reshape(-1, 1)))

### Modelling

XGBoost

In [109]:
model = XGBClassifier(
    objective = 'binary:logistic',
    eval_metric = 'logloss',
    use_label_encoder = False
)

# hyperparameter
param_grid  =  {
    'n_estimators': [100, 300],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 6],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1]
}

# grid search
grid = GridSearchCV(
                estimator = model,
                param_grid = param_grid,
                cv = 10,
                scoring = 'accuracy',
                n_jobs = -1,
                verbose = 2
)

# training
grid.fit(X_train, y_train)

Fitting 10 folds for each of 16 candidates, totalling 160 fits


d:\LEARNING\Python\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:45:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'colsample_bytree': [0.8, 1], 'learning_rate': [0.01, 0.1], 'max_depth': [3, 6], 'n_estimators': [100, 300], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",10
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter

In [110]:
best_model = grid.best_estimator_

In [111]:
y_train_pred = best_model.predict(X_train)

print(classification_report(y_train, y_train_pred))

              precision    recall  f1-score   support

           0       0.88      0.93      0.90       280
           1       0.92      0.88      0.90       280

    accuracy                           0.90       560
   macro avg       0.90      0.90      0.90       560
weighted avg       0.90      0.90      0.90       560



In [112]:
best_model = grid.best_estimator_
y_test_pred = best_model.predict(X_test)

print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.84      0.87      0.85        70
           1       0.87      0.83      0.85        70

    accuracy                           0.85       140
   macro avg       0.85      0.85      0.85       140
weighted avg       0.85      0.85      0.85       140



In [113]:
X_train

,device_Mobile,traffic_source_Ads,traffic_source_Organic,traffic_source_Social,category_Electronics,category_Fashion,category_Home,category_Sports,gender_Male,job_Engineer,...,education,pages_viewed,time_spent_sec,num_prev_transaction,Fashion_transaction_history,Home_transaction_history,Electronics_transaction_history,Sports_transaction_history,age,annual_income
287,True,False,False,True,False,True,False,False,False,False,...,1.000000,0.000000,0.746170,0.346875,0.000000,0.000000,1.0,0.000000,0.359802,0.758441
233,True,False,True,False,True,False,False,False,True,False,...,0.631511,0.254842,0.032263,0.393883,0.000000,2.293272,0.0,2.519943,0.627049,0.809727
464,True,True,False,False,True,False,False,False,False,False,...,0.000000,0.561428,0.642045,0.523711,2.573903,2.293272,0.0,2.519943,0.026712,0.261707
46,False,False,True,False,False,False,False,True,False,False,...,0.000000,0.427273,0.318712,-0.543123,0.000000,0.000000,0.0,0.000000,0.650849,0.931121
626,False,True,False,False,False,True,False,False,False,False,...,1.000000,0.853810,0.491276,0.408693,2.573903,0.000000,0.0,0.000000,0.157631,0.389721
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41,False,False,False,True,False,True,False,False,True,False,...,0.631511,0.254842,0.053443,-0.543123,0.000000,0.000000,0.0,0.000000,0.908207,0.006842
187,True,False,True,False,True,False,False,False,True,True,...,1.000000,0.254842,0.081050,-0.543123,0.000000,0.000000,0.0,0.000000,0.259695,0.248511
274,True,False,True,False,True,False,False,False,True,False,...,0.631511,0.254842,0.889537,-0.543123,0.000000,0.000000,0.0,0.000000,0.698233,0.485839
522,True,False,False,True,False,True,False,False,True,False,...,0.000000,0.561428,0.597838,0.474109,0.000000,0.000000,1.0,0.000000,0.931237,0.505008


In [114]:
scaler

{'education': MinMaxScaler(),
 'pages_viewed': MinMaxScaler(),
 'time_spent_sec': MinMaxScaler(),
 'num_prev_transaction': RobustScaler(),
 'Fashion_transaction_history': RobustScaler(),
 'Home_transaction_history': RobustScaler(),
 'Electronics_transaction_history': RobustScaler(),
 'Sports_transaction_history': RobustScaler(),
 'age': MinMaxScaler(),
 'annual_income': MinMaxScaler()}

In [115]:
def inverse_transformation(data, scaler, power) :
    for col, tool in scaler.items() :
        value = data[[col]].values
        value = scaler[col].inverse_transform(value)
        value = power[col].inverse_transform(value)

        data[col] = value.flatten()
    
    return data

In [116]:
X_train = inverse_transformation(X_train, scaler, power)
X_test = inverse_transformation(X_test, scaler, power)

In [117]:
train_data_pred = pd.concat([X_train, y_train, pd.Series(y_train_pred, index = X_train.index, name = "predictions")],  axis = 1)
test_data_pred  = pd.concat([X_test, y_test, pd.Series(y_test_pred, index = X_test.index, name = "predictions")],      axis = 1)

full_data = pd.concat([train_data_pred, test_data_pred], axis = 0)
full_data

,device_Mobile,traffic_source_Ads,traffic_source_Organic,traffic_source_Social,category_Electronics,category_Fashion,category_Home,category_Sports,gender_Male,job_Engineer,...,time_spent_sec,num_prev_transaction,Fashion_transaction_history,Home_transaction_history,Electronics_transaction_history,Sports_transaction_history,age,annual_income,purchase,predictions
287,True,False,False,True,False,True,False,False,False,False,...,385.0,660.0,0.0,0.0,1.0,0.0,32.0,14875.0,0,0
233,True,False,True,False,True,False,False,False,True,False,...,37.0,954.0,0.0,1.0,0.0,1.0,43.0,15919.0,0,0
464,True,True,False,False,True,False,False,False,False,False,...,311.0,2676.0,1.0,1.0,0.0,1.0,19.0,5810.0,1,1
46,False,False,True,False,False,False,False,True,False,False,...,132.0,0.0,0.0,0.0,0.0,0.0,44.0,18458.0,0,0
626,False,True,False,False,False,True,False,False,False,False,...,218.0,1072.0,1.0,0.0,0.0,0.0,24.0,7941.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,False,True,False,False,True,False,False,False,False,False,...,257.0,170.0,1.0,0.0,0.0,0.0,52.0,14260.0,1,0
364,True,False,False,True,False,True,False,False,True,False,...,596.0,1910.0,0.0,0.0,1.0,3.0,31.0,11692.0,1,1
292,False,False,True,False,True,False,False,False,False,False,...,297.0,5568.0,1.0,1.0,1.0,1.0,56.0,13779.0,0,0
399,True,True,False,False,True,False,False,False,False,False,...,464.0,237.0,1.0,0.0,0.0,0.0,40.0,17156.0,1,1


In [ ]:
full_data.to_csv('Result/data_with_predictions.csv')